In [1]:
from proto import *

In [2]:
disease_state_strat = Stratification("disease_state", ["S","I","R"])
humans = CompartmentMap.new(disease_state_strat)
humans.compartments

array([Compartment :[(Stratification: disease_state, 'S')],
       Compartment :[(Stratification: disease_state, 'I')],
       Compartment :[(Stratification: disease_state, 'R')]], dtype=object)

In [3]:
humans_aged,h_age_strat,remapped_comps = \
    humans.stratify(Stratification("age", ["child","young_adult","adult","older"]),in_place=False)

In [4]:
humans_aged.compartments

array([Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'young_adult')],
       Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'older')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'child')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'young_adult')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'adult')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'older')],
       Compartment :[(Stratification: disease_state, 'R'), (Stratification: age, 'child')],
       Compartment :[(Stratification: disease_state, 'R'), (Stratification: age, 'young_adult')],
       Compartment :[(Stratification: disease_state, 'R'), (St

In [5]:
non_infected = (disease_state_strat, ["S","R"])
children = (h_age_strat, ["child"])

In [6]:
cg = CompartmentGroup([non_infected, children])

In [7]:
qres = cg.query(humans_aged)
qres

CompartmentView of <proto.CompartmentMap object at 0x000002033CC17390>:
array([Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: disease_state, 'R'), (Stratification: age, 'child')]],
      dtype=object)array([0, 8])

In [10]:
qres2 = humans_aged.query([non_infected, children])
qres2

CompartmentView of <proto.CompartmentMap object at 0x000002033CC17390>:
array([Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: disease_state, 'R'), (Stratification: age, 'child')]],
      dtype=object)array([0, 8])

In [8]:
CompartmentGroup([(disease_state_strat, 'R')]).query(qres)

CompartmentView of CompartmentView of <proto.CompartmentMap object at 0x000002033CC17390>:
array([Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'child')],
       Compartment :[(Stratification: disease_state, 'R'), (Stratification: age, 'child')]],
      dtype=object)array([0, 8]):
array([Compartment :[(Stratification: disease_state, 'R'), (Stratification: age, 'child')]],
      dtype=object)array([1])

In [11]:
humans_aged.compartments[qres.indices] == qres.compartments

array([ True,  True])

In [ ]:
class TransitionFlowCompartmentMap:
    def __init__(self, source: CompartmentView, dest: CompartmentView, value_map):
        self.source = source
        self.dest = dest
        self.value_map = value_map
        self.adjustments = []

In [18]:
def remap_transition_flow_comps(flow_map, remapped_comps, map_rules=None):
    out_source = []
    out_dest = []
    out_value_map = []
    for (src_c, dest_c, value) in zip(flow_map.source, flow_map.dest, flow_map.value_map):
        new_src = remapped_comps[src_c]
        new_dest = remapped_comps[dest_c]

        if len(new_src) != len(new_dest):
            if map_rules is None:
                print(new_src, new_dest)
                raise ValueError("No flow map rules supplied for mismatched compartments")

        out_value_map.extend([value] * len(new_src))
        out_source.extend(new_src)
        out_dest.extend(new_dest)

    return TransitionFlowCompartmentMap(out_source, out_dest, out_value_map)

In [40]:
infection_orig = TransitionFlowCompartmentMap(humans.compartments[0:1],humans.compartments[1:2],["irate"])
infection_orig.source

array([Compartment :[(Stratification: disease_state, 'S')]], dtype=object)

In [41]:
infection_one_strat = remap_transition_flow_comps(infection_orig, remapped_comps)
infection_one_strat.source, infection_one_strat.dest, infection_one_strat.value_map



([Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'child')],
  Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'young_adult')],
  Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'adult')],
  Compartment :[(Stratification: disease_state, 'S'), (Stratification: age, 'older')]],
 [Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'child')],
  Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'young_adult')],
  Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'adult')],
  Compartment :[(Stratification: disease_state, 'I'), (Stratification: age, 'older')]],
 ['irate', 'irate', 'irate', 'irate'])

In [38]:
CompartmentGroup([(h_age_strat, "child")]).query(infection_one_strat.source)

AttributeError: 'list' object has no attribute 'compartments'

In [ ]:
humans_sev,severity_strat,sev_remapped_comps = humans.stratify(Stratification("severity", ["asymp", "mild", "severe"]), (disease_state_strat, "I"), in_place=False)
humans_sev.compartments

array([Compartment :[(Stratification: disease_state, 'S')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: severity, 'asymp')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: severity, 'mild')],
       Compartment :[(Stratification: disease_state, 'I'), (Stratification: severity, 'severe')],
       Compartment :[(Stratification: disease_state, 'R')]], dtype=object)

In [ ]:
infection_sev_strat = remap_transition_flow_comps(infection_orig, sev_remapped_comps)

[Compartment :[(Stratification: disease_state, 'S')]] [Compartment :[(Stratification: disease_state, 'I'), (Stratification: severity, 'asymp')], Compartment :[(Stratification: disease_state, 'I'), (Stratification: severity, 'mild')], Compartment :[(Stratification: disease_state, 'I'), (Stratification: severity, 'severe')]]


ValueError: No flow map rules supplied for mismatched compartments

In [ ]:
cat_indices = [base_cg.query(c).indices for c in infection_cats]

def category_idx_reduction(cat_indices: list[np.ndarray], src: jax.Array):
    if len(set([len(c) for c in cat_indices])) == 1:
        return src[np.array(cat_indices)].sum(axis=1)
    else:
        return jnp.array([src[c].sum() for c in cat_indices])
